In [3]:
import os
import pandas as pd
import numpy as np
import hashlib
import random
import json
import base64

def prepare(public_dir='./public', private_dir='./private'):
    print("Initializing Polymorphic Cipher Engine...")
    
    # Enforce determinism
    np.random.seed(42)
    random.seed(42)
    
    # Staying strictly aligned with the deep-space/interstellar theme
    concepts = ['alpha', 'beta', 'gamma', 'delta', 'epsilon', 'zeta']
    raw_data = []
    
    for i in range(1000):
        p_concepts = random.sample(concepts, 3)
        logic_type = random.choice(['entailment', 'contradiction', 'neutral'])
        
        if logic_type == 'entailment':
            h_concepts = random.sample(p_concepts, random.randint(1, 3))
            state = "SYNC-OK"
        elif logic_type == 'contradiction':
            h_concepts = [random.choice(p_concepts), 'alert_override']
            state = "SYNC-ERR"
        else: 
            h_concepts = random.sample([c for c in concepts if c not in p_concepts], 2)
            state = "SYNC-IDLE"
            
        raw_data.append({"premise": p_concepts, "hypothesis": h_concepts, "state": state})
        
    df = pd.DataFrame(raw_data)
    
    contexts = []
    truths = []
    
    print("Scrambling arrays and injecting interstellar radiation noise...")
    for idx, row in df.iterrows():
        uid = hashlib.md5(f"seq_{idx}_{''.join(row['premise'])}".encode('utf-8')).hexdigest()
        
        p_tokens = [f"PRM-{hashlib.md5((c + 'salt_1').encode()).hexdigest()[:4].upper()}" for c in row['premise']]
        h_tokens = [f"SEC-{hashlib.md5((c + 'salt_2').encode()).hexdigest()[:4].upper()}" for c in row['hypothesis']]
        
        if idx % 2 == 0: p_tokens.append("RAD-E499")
        if idx % 3 == 0: h_tokens.append("RAD-88FX")
            
        p_tokens.sort()
        h_tokens.sort()
        
        contexts.append({
            "id": uid, 
            "primary_array": " ".join(p_tokens),
            "secondary_array": " ".join(h_tokens)
        })
        truths.append({"id": uid, "sync_state": row['state']})
        
    split_idx = int(len(contexts) * 0.8)
    df_train = pd.DataFrame(contexts[:split_idx])
    df_train['sync_state'] = [t['sync_state'] for t in truths[:split_idx]]
    
    df_test = pd.DataFrame(contexts[split_idx:])
    df_truth = pd.DataFrame(truths[split_idx:])
    
    # Create required directories
    os.makedirs(public_dir, exist_ok=True)
    os.makedirs(private_dir, exist_ok=True)
    
    # 1. Output Public Files (Train, Test, Sample Sub)
    print("Writing files to /public...")
    df_train.to_parquet(os.path.join(public_dir, "train.parquet"), index=False)
    df_test.to_parquet(os.path.join(public_dir, "test.parquet"), index=False)
    
    sample_sub = df_test[['id']].copy()
    sample_sub['sync_state'] = "TK-0000"
    if len(sample_sub) > 0:
        first_id = sample_sub.iloc[0]['id']
        correct_ans = df_truth[df_truth['id'] == first_id]['sync_state'].values[0]
        sample_sub.loc[0, 'sync_state'] = correct_ans
    sample_sub.to_csv(os.path.join(public_dir, "sample_submission.csv"), index=False)
    
    # 2. Generate the Ghost Grader
    print("Encoding Ground Truth and generating the Ghost Grader...")
    
    # Create a dictionary of {id: sync_state} and encode it
    truth_dict = dict(zip(df_truth['id'], df_truth['sync_state']))
    truth_json = json.dumps(truth_dict)
    truth_b64 = base64.b64encode(truth_json.encode('utf-8')).decode('utf-8')
    
    # The Ghost Grader Script Template
    grader_code = f'''import pandas as pd
import json
import base64
import os

# Embedded Base64 Ground Truth (Ghost Grader)
EMBEDDED_TRUTH = "{truth_b64}"

def grade(sub_input, truth_input=None):
    try:
        # Decode the hidden internal memory
        truth_dict = json.loads(base64.b64decode(EMBEDDED_TRUTH).decode('utf-8'))
        
        # Robust handling for submission input
        if isinstance(sub_input, str):
            if os.path.isdir(sub_input):
                sub_input = os.path.join(sub_input, "submission.csv")
            sub_df = pd.read_csv(sub_input)
        else:
            sub_df = sub_input.copy()

        # Clean strings for exact matching
        sub_df['sync_state'] = sub_df['sync_state'].astype(str).str.strip().str.upper()

        # Grade against internal dictionary
        matches = 0
        for _, row in sub_df.iterrows():
            sub_id = str(row['id']).strip()
            sub_state = row['sync_state']
            if sub_id in truth_dict and truth_dict[sub_id] == sub_state:
                matches += 1
        
        accuracy = matches / len(truth_dict)
        return float(accuracy)
        
    except Exception:
        # Failsafe to prevent platform crashes
        return 0.0
'''
    # Write the custom grader script
    with open(os.path.join(private_dir, "grade.py"), "w") as f:
        f.write(grader_code)
        
    # Write a dummy answers.csv just in case the UI demands a file to save the form
    dummy_df = pd.DataFrame(columns=['id', 'sync_state'])
    dummy_df.to_csv(os.path.join(private_dir, "dummy_answers.csv"), index=False)

    print("Done! Architecture is locked and ready for deployment.")

if __name__ == "__main__":
    prepare()

Initializing Polymorphic Cipher Engine...
Scrambling arrays and injecting interstellar radiation noise...
Writing files to /public...
Encoding Ground Truth and generating the Ghost Grader...
Done! Architecture is locked and ready for deployment.
